<a href="https://www.kaggle.com/code/saibhossain/document-aware-hybrid-rag?scriptVersionId=311337778" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# What we are changing

basic:

    documents -> chunks -> FAISS retriever -> LLM

update :

    documents -> chunks
              -> document catalog
              -> document selection (paper-level)
              -> filtered chunk set
              -> BM25 + FAISS hybrid retrieval
              -> LLM


---

This solves your main failure:

* mixed-topic PDFs
* vague paper references
* wrong paper chosen before answering

# old code  

In [ ]:
!pip -q install langchain langchain-community langchain-core
!pip -q install langchain-huggingface langchain-google-genai
!pip -q install sentence-transformers faiss-cpu pypdf rank-bm25
!pip install -q langchain-classic #

In [ ]:
import os
import re
import glob
import json
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional
from collections import defaultdict # new add

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever

In [ ]:
@dataclass
class RAGConfig:
    pdf_dir: str = "/kaggle/input/datasets/saibhossain/rag-practice"
    embedding_model_name: str = "sentence-transformers/all-MiniLM-L6-v2"
    llm_model_name: str = "gemini-2.5-flash"

    chunk_size: int = 800
    chunk_overlap: int = 150

    dense_top_k: int = 6
    sparse_top_k: int = 6
    final_top_k: int = 5

    dense_weight: float = 0.55
    sparse_weight: float = 0.45

    # new
    doc_select_top_k: int = 3
    min_docs_for_compare: int = 2
    query_preview_chars: int = 180

    temperature: float = 0.2

config = RAGConfig()
print(config)

## update PDF loading

In [ ]:
def normalize_filename(name: str) -> str:
    base = os.path.basename(name)
    base = base.replace(".pdf", "")
    base = re.sub(r"[_\-]+", " ", base)
    base = re.sub(r"\s+", " ", base).strip()
    return base.lower()


def get_pdf_paths(pdf_dir: str) -> List[str]:
    pdf_paths = glob.glob(os.path.join(pdf_dir, "*.pdf"))
    pdf_paths = sorted(pdf_paths)
    return pdf_paths


def load_pdfs_from_folder(pdf_dir: str) -> List[Document]:
    pdf_paths = get_pdf_paths(pdf_dir)

    if not pdf_paths:
        raise FileNotFoundError(f"No PDF files found in: {pdf_dir}")

    print("=" * 100)
    print("STEP 1: LOADING PDF FILES")
    print("=" * 100)
    print(f"Found {len(pdf_paths)} PDF files\n")

    all_docs = []

    for pdf_path in pdf_paths:
        file_name = os.path.basename(pdf_path)
        file_stem = file_name.replace(".pdf", "")

        print(f"Loading: {file_name}")
        loader = PyPDFLoader(pdf_path)
        docs = loader.load()

        for doc in docs:
            doc.metadata["file_name"] = file_name
            doc.metadata["file_stem"] = file_stem
            doc.metadata["normalized_file_name"] = normalize_filename(file_name)
            doc.metadata["source_path"] = pdf_path

        print(f"  -> Pages loaded: {len(docs)}")
        if docs:
            preview = docs[0].page_content[:250].replace("\n", " ")
            print(f"  -> First page preview: {preview}")
        print("-" * 100)

        all_docs.extend(docs)

    print(f"\nTotal pages loaded across all PDFs: {len(all_docs)}")
    return all_docs


documents = load_pdfs_from_folder(config.pdf_dir)

print("\nSample metadata:")
print(documents[0].metadata)

print("\nSample page text preview:")
print(documents[0].page_content[:1000])

## Update Chunking 

In [ ]:
def chunk_documents(documents: List[Document], chunk_size: int, chunk_overlap: int) -> List[Document]:
    print("=" * 100)
    print("STEP 2: CHUNKING DOCUMENTS")
    print("=" * 100)

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    chunks = text_splitter.split_documents(documents)

    file_chunk_counter = defaultdict(int)

    for i, chunk in enumerate(chunks):
        file_name = chunk.metadata.get("file_name", "unknown_file")
        chunk.metadata["global_chunk_id"] = i
        chunk.metadata["chunk_id"] = file_chunk_counter[file_name]
        chunk.metadata["chunk_len"] = len(chunk.page_content)
        file_chunk_counter[file_name] += 1

        chunk.metadata["doc_key"] = chunk.metadata.get("normalized_file_name", normalize_filename(file_name))

    print(f"Total chunks created: {len(chunks)}")

    if chunks:
        chunk_lengths = [len(c.page_content) for c in chunks]
        print(f"Min chunk length: {min(chunk_lengths)}")
        print(f"Max chunk length: {max(chunk_lengths)}")
        print(f"Average chunk length: {sum(chunk_lengths)/len(chunk_lengths):.2f}")

    print("\nExample chunk metadata:")
    print(chunks[0].metadata)

    print("\nExample chunk text:")
    print(chunks[0].page_content[:1000])

    return chunks


chunks = chunk_documents(
    documents=documents,
    chunk_size=config.chunk_size,
    chunk_overlap=config.chunk_overlap
)

In [ ]:
def build_embedding_model(model_name: str):
    print("=" * 100)
    print("STEP 3: LOADING EMBEDDING MODEL")
    print("=" * 100)

    embedding_model = HuggingFaceEmbeddings(model_name=model_name)
    print(f"Embedding model loaded: {model_name}")
    return embedding_model


embedding_model = build_embedding_model(config.embedding_model_name)

In [ ]:
def build_vectorstore(chunks: List[Document], embedding_model) -> FAISS:
    print("=" * 100)
    print("STEP 4: BUILDING FAISS VECTOR STORE")
    print("=" * 100)

    vectorstore = FAISS.from_documents(
        documents=chunks,
        embedding=embedding_model
    )

    print("FAISS vector store created successfully")
    print(f"Indexed chunk count: {len(chunks)}")
    return vectorstore


vectorstore = build_vectorstore(chunks, embedding_model)

# new 

### build a paper-level catalog

In [ ]:
def clean_preview_text(text: str, max_chars: int = 1200) -> str:
    text = re.sub(r"\s+", " ", text).strip()
    return text[:max_chars]


def build_document_catalog(documents: List[Document]) -> List[Dict[str, Any]]:
    print("=" * 100)
    print("STEP 5: BUILDING DOCUMENT CATALOG")
    print("=" * 100)

    grouped = defaultdict(list)
    for doc in documents:
        file_name = doc.metadata["file_name"]
        grouped[file_name].append(doc)

    catalog = []

    for file_name, docs in grouped.items():
        docs = sorted(docs, key=lambda d: d.metadata.get("page", 0))
        first_page_text = docs[0].page_content if docs else ""
        preview_text = clean_preview_text(first_page_text, max_chars=1500)

        entry = {
            "file_name": file_name,
            "file_stem": docs[0].metadata.get("file_stem", file_name.replace(".pdf", "")),
            "normalized_file_name": docs[0].metadata.get("normalized_file_name", normalize_filename(file_name)),
            "page_count": len(docs),
            "preview_text": preview_text,
            "document_repr": f"TITLE: {file_name}\nPREVIEW: {preview_text}"
        }
        catalog.append(entry)

    catalog = sorted(catalog, key=lambda x: x["file_name"])

    print(f"Built catalog for {len(catalog)} papers\n")
    for i, item in enumerate(catalog[:5], start=1):
        print(f"{i}. {item['file_name']} | pages={item['page_count']}")
        print(f"   Preview: {item['preview_text'][:200]}")
        print("-" * 100)

    return catalog


document_catalog = build_document_catalog(documents)

## build document selector

In [ ]:
def build_document_selector(document_catalog: List[Dict[str, Any]]):
    print("=" * 100)
    print("STEP 6: BUILDING DOCUMENT SELECTOR")
    print("=" * 100)

    selector_docs = []
    for item in document_catalog:
        selector_docs.append(
            Document(
                page_content=item["document_repr"],
                metadata={
                    "file_name": item["file_name"],
                    "normalized_file_name": item["normalized_file_name"],
                    "page_count": item["page_count"]
                }
            )
        )

    doc_selector_bm25 = BM25Retriever.from_documents(selector_docs)
    doc_selector_bm25.k = config.doc_select_top_k

    print("Document selector BM25 built successfully")
    print(f"Document selector top_k: {config.doc_select_top_k}")
    return selector_docs, doc_selector_bm25


selector_docs, doc_selector_bm25 = build_document_selector(document_catalog)

## helper functions for document-aware retrieval

In [ ]:
def query_mentions_filename(query: str, file_name: str) -> bool:
    q = normalize_filename(query)
    f = normalize_filename(file_name)

    if f in q:
        return True

    file_stem = f.replace(".pdf", "").strip()
    if len(file_stem) > 8 and file_stem in q:
        return True

    return False


def infer_compare_mode(query: str) -> bool:
    q = query.lower()
    compare_keywords = ["compare", "difference", "different papers", "versus", "vs"]
    return any(k in q for k in compare_keywords)


def select_candidate_documents(query: str,
                               document_catalog: List[Dict[str, Any]],
                               doc_selector_bm25,
                               top_k: int = 3) -> List[str]:
    q = query.lower()

    # 1. explicit filename match first
    exact_matches = []
    for item in document_catalog:
        if query_mentions_filename(q, item["file_name"]):
            exact_matches.append(item["file_name"])

    if exact_matches:
        return exact_matches[:top_k]

    # 2. BM25 over document titles + preview
    selected_docs = doc_selector_bm25.invoke(query)
    selected_file_names = []
    for d in selected_docs:
        name = d.metadata["file_name"]
        if name not in selected_file_names:
            selected_file_names.append(name)

    # 3. for compare queries, try to keep at least 2 docs
    if infer_compare_mode(query) and len(selected_file_names) < config.min_docs_for_compare:
        for item in document_catalog:
            if item["file_name"] not in selected_file_names:
                selected_file_names.append(item["file_name"])
            if len(selected_file_names) >= config.min_docs_for_compare:
                break

    return selected_file_names[:top_k]

In [ ]:
def get_chunks_for_selected_docs(chunks: List[Document], selected_file_names: List[str]) -> List[Document]:
    selected_set = set(selected_file_names)
    filtered_chunks = [c for c in chunks if c.metadata.get("file_name") in selected_set]
    return filtered_chunks


def build_filtered_hybrid_retriever(filtered_chunks: List[Document],
                                    embedding_model,
                                    dense_top_k: int,
                                    sparse_top_k: int,
                                    dense_weight: float,
                                    sparse_weight: float):
    if not filtered_chunks:
        raise ValueError("No filtered chunks available to build hybrid retriever.")

    local_vectorstore = FAISS.from_documents(filtered_chunks, embedding_model)

    dense_retriever = local_vectorstore.as_retriever(
        search_kwargs={"k": min(dense_top_k, len(filtered_chunks))}
    )

    sparse_retriever = BM25Retriever.from_documents(filtered_chunks)
    sparse_retriever.k = min(sparse_top_k, len(filtered_chunks))

    hybrid_retriever = EnsembleRetriever(
        retrievers=[sparse_retriever, dense_retriever],
        weights=[sparse_weight, dense_weight]
    )

    return dense_retriever, sparse_retriever, hybrid_retriever

In [ ]:
def print_selected_documents(selected_file_names: List[str]):
    print("\n" + "=" * 120)
    print("DOCUMENT SELECTION DEBUG")
    print("=" * 120)

    if not selected_file_names:
        print("No documents selected.")
        return

    for i, name in enumerate(selected_file_names, start=1):
        print(f"{i}. {name}")


def inspect_single_retriever(query: str, retriever, retriever_name: str) -> List[Document]:
    print("\n" + "=" * 120)
    print(f"RETRIEVER DEBUG: {retriever_name}")
    print("=" * 120)
    print(f"Query: {query}\n")

    docs = retriever.invoke(query)
    print(f"Retrieved {len(docs)} chunks\n")

    for i, doc in enumerate(docs, start=1):
        print("-" * 120)
        print(f"Rank: {i}")
        print(f"Chunk ID: {doc.metadata.get('chunk_id', 'unknown')}")
        print(f"Global Chunk ID: {doc.metadata.get('global_chunk_id', 'unknown')}")
        print(f"File: {doc.metadata.get('file_name', 'unknown')}")
        print(f"Page: {doc.metadata.get('page', 'unknown')}")
        print(f"Chunk Length: {doc.metadata.get('chunk_len', 'unknown')}")
        print("Preview:")
        print(doc.page_content[:800])
        print()

    return docs

In [ ]:
def format_context(retrieved_docs: List[Document]) -> str:
    context_parts = []

    for i, doc in enumerate(retrieved_docs, start=1):
        source = doc.metadata.get("file_name", "unknown_file")
        page = doc.metadata.get("page", "unknown_page")
        chunk_id = doc.metadata.get("chunk_id", "unknown_chunk")
        text = doc.page_content.strip()

        context_parts.append(
            f"[Chunk {i} | Chunk ID: {chunk_id} | Source: {source} | Page: {page}]\n{text}"
        )

    return "\n\n".join(context_parts)

In [ ]:
def build_rag_prompt(query: str, retrieved_docs: List[Document], selected_file_names: List[str]) -> str:
    context = format_context(retrieved_docs)
    selected_docs_text = "\n".join([f"- {x}" for x in selected_file_names])

    prompt = f"""
You are a careful research assistant.

Your task:
1. Answer the question using ONLY the retrieved context.
2. The system has already selected the most relevant candidate papers.
3. Do not mix in information from papers outside the selected candidate papers.
4. If the query refers to a specific paper but the identity is still ambiguous, clearly say so.
5. If the answer is not available in the retrieved context, say:
   "I could not find the answer in the provided context."
6. Always cite file name and page number when possible.
7. For comparison questions, compare only papers actually supported by the retrieved context.

Selected Candidate Papers:
{selected_docs_text}

Retrieved Context:
{context}

User Question:
{query}

Answer:
"""
    return prompt.strip()

In [ ]:
def ask_document_aware_hybrid_rag(query: str,
                                  chunks: List[Document],
                                  document_catalog: List[Dict[str, Any]],
                                  doc_selector_bm25,
                                  embedding_model,
                                  llm,
                                  config: RAGConfig,
                                  debug: bool = True) -> Dict[str, Any]:

    # Step A: select relevant papers
    selected_file_names = select_candidate_documents(
        query=query,
        document_catalog=document_catalog,
        doc_selector_bm25=doc_selector_bm25,
        top_k=config.doc_select_top_k
    )

    if debug:
        print_selected_documents(selected_file_names)

    # Step B: keep only chunks from selected papers
    filtered_chunks = get_chunks_for_selected_docs(chunks, selected_file_names)

    if debug:
        print("\n" + "=" * 120)
        print("FILTERED CHUNK DEBUG")
        print("=" * 120)
        print(f"Selected papers: {len(selected_file_names)}")
        print(f"Filtered chunks available: {len(filtered_chunks)}")

    if not filtered_chunks:
        return {
            "query": query,
            "selected_file_names": selected_file_names,
            "retrieved_docs": [],
            "prompt": "",
            "answer": "I could not find the answer in the provided context."
        }

    # Step C: build local hybrid retriever only on selected docs
    dense_retriever, sparse_retriever, hybrid_retriever = build_filtered_hybrid_retriever(
        filtered_chunks=filtered_chunks,
        embedding_model=embedding_model,
        dense_top_k=config.dense_top_k,
        sparse_top_k=config.sparse_top_k,
        dense_weight=config.dense_weight,
        sparse_weight=config.sparse_weight
    )

    if debug:
        sparse_docs = inspect_single_retriever(query, sparse_retriever, "FILTERED SPARSE BM25")
        dense_docs = inspect_single_retriever(query, dense_retriever, "FILTERED DENSE FAISS")
        hybrid_docs = inspect_single_retriever(query, hybrid_retriever, "FILTERED HYBRID ENSEMBLE")
    else:
        hybrid_docs = hybrid_retriever.invoke(query)

    retrieved_docs = hybrid_docs[:config.final_top_k]
    prompt = build_rag_prompt(query, retrieved_docs, selected_file_names)
    response = llm.invoke(prompt)

    return {
        "query": query,
        "selected_file_names": selected_file_names,
        "filtered_chunk_count": len(filtered_chunks),
        "retrieved_docs": retrieved_docs,
        "prompt": prompt,
        "answer": response.content
    }

In [ ]:
def print_rag_result(result: Dict[str, Any]):
    print("=" * 120)
    print("QUERY")
    print("=" * 120)
    print(result["query"])

    print("\n" + "=" * 120)
    print("SELECTED PAPERS")
    print("=" * 120)
    for i, name in enumerate(result.get("selected_file_names", []), start=1):
        print(f"{i}. {name}")

    print("\n" + "=" * 120)
    print("ANSWER")
    print("=" * 120)
    print(result["answer"])

    print("\n" + "=" * 120)
    print("RETRIEVED SOURCES")
    print("=" * 120)

    for i, doc in enumerate(result["retrieved_docs"], start=1):
        print(f"\nRank {i}")
        print(f"Chunk ID: {doc.metadata.get('chunk_id', 'unknown')}")
        print(f"Global Chunk ID: {doc.metadata.get('global_chunk_id', 'unknown')}")
        print(f"File: {doc.metadata.get('file_name', 'unknown')}")
        print(f"Page: {doc.metadata.get('page', 'unknown')}")
        print(f"Chunk Length: {doc.metadata.get('chunk_len', 'unknown')}")
        print(f"Preview: {doc.page_content[:500]}")

## llm

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
os.environ["GOOGLE_API_KEY"] = user_secrets.get_secret("GOOGLE_API_KEY")

def build_llm(model_name: str, temperature: float):
    print("=" * 100)
    print("STEP 7: LOADING LLM")
    print("=" * 100)

    llm = ChatGoogleGenerativeAI(
        model=model_name,
        temperature=temperature
    )

    print(f"LLM initialized successfully: {model_name}")
    return llm


llm = build_llm(config.llm_model_name, config.temperature)

## llm with qroq

In [ ]:
!pip -q install -U langchain-groq

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ["GROQ_API_KEY"] = user_secrets.get_secret("groq-API")

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq



def build_llm(model_name: str, temperature: float):
    print("=" * 100)
    print("STEP 7: LOADING LLM")
    print("=" * 100)

    llm = ChatGroq(
        model="openai/gpt-oss-120b",
        temperature=temperature,
        max_retries=2,
        timeout=60,
    )

    print(f"LLM initialized successfully: {model_name}")
    return llm


llm = build_llm(config.llm_model_name, config.temperature)

In [ ]:
test = llm.invoke("Say hello in one short sentence.")
print(test.content)

# test

## one

In [ ]:
result = ask_document_aware_hybrid_rag(
    query="What dataset was used in the Lung Cancer Detection paper?",
    chunks=chunks,
    document_catalog=document_catalog,
    doc_selector_bm25=doc_selector_bm25,
    embedding_model=embedding_model,
    llm=llm,
    config=config,
    debug=True
)

print_rag_result(result)

## evaluation

In [ ]:
queries = [
    "What problem does the Lung Cancer Detection paper try to solve?",
    "What dataset was used in the Lung Cancer Detection paper?",
    "What methods are used in the Deep learning-based approach to diagnose lung cancer using CT-scan images paper?",
    "Which paper discusses survival prediction models?",
    "What limitations are mentioned in LungPaper.pdf?",
    "Compare the methods used in LungPaper.pdf and Deep learning-based approach to diagnose lung cancer using CT-scan images.pdf"
]

for q in queries:
    print("\n\n" + "#" * 120)

    result = ask_document_aware_hybrid_rag(
        query=q,
        chunks=chunks,
        document_catalog=document_catalog,
        doc_selector_bm25=doc_selector_bm25,
        embedding_model=embedding_model,
        llm=llm,
        config=config,
        debug=True
    )

    print_rag_result(result)

---

# Failure table of Document-aware Hybrid RAG

| Query                                                                                                             | Outcome                  | Failure level | What happened                                                                                                                                                                                                                                  | Main reason                                                                                                                                                           |
| ----------------------------------------------------------------------------------------------------------------- | ------------------------ | ------------: | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | --------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **What problem does the Lung Cancer Detection paper try to solve?**                                               | Answered reasonably well | Low / partial | It selected 3 lung-cancer papers and answered using evidence from more than one of them, including the AI-Powered paper and the Deep learning-based paper.                                                                                     | The system still could not fully decide which single paper “the Lung Cancer Detection paper” meant.                                                                   |
| **What dataset was used in the Lung Cancer Detection paper?**                                                     | Failed                   |          High | Retrieval actually found dataset evidence from `1-s2.0...` and `fmed...`, including LIDC-IDRI and Lung-PET-CT-Dx, but final answer said it could not find the answer.                                                                          | The system had evidence, but the question remained paper-ambiguous, so the generator became overly conservative.                                                      |
| **What methods are used in the Deep learning-based approach to diagnose lung cancer using CT-scan images paper?** | Good                     |           Low | It selected only one paper and retrieved highly relevant chunks describing CNNs, CDL, DenseNet201 features, ConvNeXt, and evaluation methods.                                                                                                  | Exact paper grounding removed ambiguity. This is where Document-aware Hybrid RAG works best.                                                                          |
| **Which paper discusses survival prediction models?**                                                             | Good                     | Low / partial | It selected 3 papers, but top evidence clearly pointed to `Deep learning-based approach...` through the Doppalapudi citation on survival prediction models.                                                                                    | Retrieval succeeded because one chunk explicitly matched the phrase “survival prediction models.”                                                                     |
| **What limitations are mentioned in LungPaper.pdf?**                                                              | Failed                   |        Medium | It selected only `LungPaper.pdf`, which is good, and retrieved several limitation-like chunks about imperfections, overfitting, irregular tumor shapes, and texture challenges. But the final answer still said it could not find the answer.  | Retrieval was good, but answer synthesis failed because “limitations” were expressed implicitly across several chunks rather than one explicit “Limitations” section. |
| **Compare the methods used in LungPaper.pdf and Deep learning-based approach to diagnose...**                     | Failed                   |          High | It selected the correct two papers, but most top-ranked chunks came from `Deep learning-based...`; only one good `LungPaper.pdf` method chunk appeared lower down. Final answer said it could not find the answer.                             | Cross-paper balance failed. The retriever favored one paper instead of ensuring evidence coverage from both papers.                                                   |


## Compare: Hybrid RAG vs Document-aware Hybrid RAG

| Aspect                               | Hybrid RAG            | Document-aware Hybrid RAG                                 |
| ------------------------------------ | --------------------- | --------------------------------------------------------- |
| Retrieval scope                      | Whole corpus          | Selected candidate papers only                            |
| Cross-topic contamination            | High                  | Much lower                                                |
| Works on mixed-topic PDFs            | Weak                  | Better                                                    |
| Handles exact paper name queries     | Medium                | Strong                                                    |
| Handles vague paper references       | Weak                  | Medium                                                    |
| Handles paper-level dataset question | Weak                  | Medium, but still fragile                                 |
| Handles implicit limitations         | Weak                  | Medium, retrieval better but synthesis still weak         |
| Handles two-paper comparison         | Weak                  | Better paper selection, but still poor comparison         |
| Main failure mode                    | Wrong topic retrieved | Right topic, but wrong paper resolution or weak synthesis |


---------

# 👨‍💻 Author
# **Md Saib Hossain**
**AI Engineer • AI / ML / LLM & AI Safety Researcher**  
**Agentic AI Developer • Researcher in Autonomous & Multi-Agent Systems • Advanced Agentic AI Architect**

Designing safe, scalable, and human-centered intelligent systems for real-world healthcare and autonomous AI applications.

<p align="left">
  <a href="mailto:saibhossain5@gmail.com">
    <img src="https://img.shields.io/badge/Email-saibhossain5%40gmail.com-red?style=flat&logo=gmail">
  </a>
  <a href="https://saibhossain.github.io/">
    <img src="https://img.shields.io/badge/Portfolio-Visit-blue?style=flat&logo=google-chrome">
  </a>
  <a href="https://github.com/Saibhossain">
    <img src="https://img.shields.io/badge/GitHub-Profile-black?style=flat&logo=github">
  </a>
  <a href="https://linkedin.com/in/saib-hossain-182834229">
    <img src="https://img.shields.io/badge/LinkedIn-Connect-0A66C2?style=flat&logo=linkedin">
  </a>
</p>
